In [1]:
import numpy as np

def volume_from_grid_axes(grid, axes):
    """
    Convert our (nx, ny, nz) grid + 1D axes into napari-friendly stuff.
    napari expects (Z, Y, X) array plus an affine or (scale, translate) for world coords.

    Returns
    -------
    volume : (nz, ny, nx) ndarray
    scale  : (dz, dy, dx) voxel size along z, y, x
    translate : (z0, y0, x0) world coord of voxel (0,0,0)
    is_uniform : bool (whether axes are strictly uniform; if False we used average spacing)
    """
    xax, yax, zax = [np.asarray(a) for a in axes]   # lengths nx, ny, nz
    nx, ny, nz = len(xax), len(yax), len(zax)
    assert grid.shape == (nx, ny, nz)

    # napari expects zyx ordering
    volume = grid.transpose(2, 1, 0).copy()  # -> (nz, ny, nx)

    # compute spacing; if non-uniform, use average (napari only supports linear transforms)
    def _spacing(a):
        d = np.diff(a)
        return float(d.mean()), bool(np.allclose(d, d[0], rtol=1e-5, atol=1e-8))
    dx, x_uni = _spacing(xax)
    dy, y_uni = _spacing(yax)
    dz, z_uni = _spacing(zax)
    is_uniform = x_uni and y_uni and z_uni

    # scale is (dz, dy, dx) for (Z,Y,X)
    scale = (dz, dy, dx)
    translate = (float(zax[0]), float(yax[0]), float(xax[0]))
    return volume, scale, translate, is_uniform


def log1p_clip(a):
    """Nice-looking intensity for volume rendering."""
    a = np.asarray(a)
    a = np.maximum(a, 0.0)
    return np.log1p(a)

In [2]:
from rsm3d.rsm3d import RSMBuilder  # <-- 4-circle xrayutilities builder
spec_file = "/nsls2/data/staff/xyang4/data_cs/isr_rsm3d/setup_6oct23"
tiff_dir  = "/nsls2/data/staff/xyang4/data_cs/isr_rsm3d/data_6oct23_tiff"
scan_list = (17, 18, 19)     # any list/tuple of scan numbers
out_vtr   = "/nsls2/data/staff/xyang4/data_cs/isr_rsm3d/rsm_hkl.vtr"    # output file

# Build with 4-circle (ZXZ: φ(Z) → χ(X) → ω(Z))
builder = RSMBuilder(
    spec_file, tiff_dir,
    selected_scans=scan_list,
    ub_includes_2pi=True,        # set False if your UB is "no-2π"
    center_is_one_based=False,   # True if SPEC xcenter/ycenter are 1-based
    fourc_mode="ZXZ",            # or "ZYX" if your instrument uses Z-Y-X
    motor_map={"omega":"th", "chi":"chi", "phi":"phi"},  # map to your SPEC columns
)

# Compute per-pixel Q & HKL
Q_samp, hkl, intensity = builder.compute_full()

# Regrid with xrayutilities gridder (mean or sum)
grid, (xax, yax, zax) = builder.regrid_xu(
    space="hkl",                 # "hkl" or "q"
    grid_shape=(256, 256, 256),  # adjust to taste / memory
    ranges=None,                 # or ((xmin,xmax),(ymin,ymax),(zmin,zmax)) to lock axes
    fuzzy=False,                 # True → FuzzyGridder3D; add width=... for footprint
    normalize="mean",            # "mean" or "sum"
    stream=True                  # frame-by-frame accumulation (RAM friendly)
)
grid.shape, len(xax), len(yax), len(zax)

((256, 256, 256), 256, 256, 256)

In [3]:
# Assume you've already done:
# grid, (xax, yax, zax) = builder.regrid_xu(space="hkl", grid_shape=(256,256,256), normalize="mean", stream=True)

import napari

volume, scale, translate, is_uniform = volume_from_grid_axes(grid, (xax, yax, zax))
v = napari.Viewer(ndisplay=3)
v.add_image(
    log1p_clip(volume),               # 3D image (Z,Y,X)
    name="RSM (log1p)", 
    colormap="viridis",
    rendering="attenuated_mip",       # alternatives: 'mip', 'translucent'
    opacity=1.0,
    scale=scale,                      # voxel spacing
    translate=translate               # world origin
)

# Optional: also add the linear (non-log) version
# v.add_image(volume, name="RSM (linear)", colormap="magma", rendering="mip", scale=scale, translate=translate)

print("Uniform spacing:", is_uniform)  # if False, we used average spacing (linear approx)

Uniform spacing: True
